# M6.1 · Autograd internals

`loss.backward()` çağrınca perde arkasında ne oluyor? Hook'lu bir hesap grafında gradyanları elle izliyor, saf Python'da bir grafı topolojik sıralayıp yalnız yapraklara gradyan biriktiren bir motor yazıyor, NumPy'da PyTorch'un "version counter" mekanizmasını kurup hangi in-place işlemlerin geri geçişi bozduğunu görüyor, PyTorch'ta `grad_fn` grafını geziyor, saklanan tensörleri sayıyor, tensor hook'u yazıyor ve en sonda saklanan aktivasyonları bf16'ya sıkıştırarak belleği yarıya indiriyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m6-1-autograd-internals.html](https://m-gonul.github.io/pytorch-lab/m6-1-autograd-internals.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m6_1_autograd_internals.py
CHECKS = {
 "b": "import random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    g = {\"a\": (\"leaf\", []), \"b\": (\"leaf\", []), \"c\": (\"mul\", [\"a\", \"b\"]), \"d\": (\"add\", [\"c\", \"a\"]), \"L\": (\"square\", [\"d\"])}\n\n    def fw():\n        v = need(ns, \"forward\")(g, {\"a\": 2.0, \"b\": 3.0})\n        ensure(v[\"L\"] == 64 and v[\"d\"] == 8, f\"değerler {v}\")\n    R.test(\"forward\", fw)\n\n    def ts():\n        order = need(ns, \"toposort\")(g, \"L\")\n        ensure(order[-1] == \"L\" and set(order) == set(g), f\"sıra {order}\")\n        for n, (_, ins) in g.items():\n            for i in ins:\n                ensure(order.index(i) < order.index(n), f\"{i}, {n}'den önce gelmeli\")\n        o2 = ns[\"toposort\"](g, \"c\")\n        ensure(set(o2) == {\"a\", \"b\", \"c\"}, \"yalnız erişilebilir düğümler\")\n    R.test(\"toposort\", ts)\n\n    def bw():\n        v = ns[\"forward\"](g, {\"a\": 2.0, \"b\": 3.0})\n        ensure(need(ns, \"backward\")(g, v, \"L\") == {\"a\": 64.0, \"b\": 32.0}, \"hook'suz\")\n        ensure(ns[\"backward\"](g, v, \"L\", hooks={\"c\": lambda x: 2 * x}) == {\"a\": 112.0, \"b\": 64.0}, \"c'de ×2 hook\")\n        ensure(ns[\"backward\"](g, v, \"L\", hooks={\"a\": lambda x: 0.0}) == {\"a\": 0.0, \"b\": 32.0}, \"yaprak hook'u\")\n    R.test(\"backward + hook'lar\", bw)\n\n    def deep():\n        rng = random.Random(0)\n        G = {\"x\": (\"leaf\", []), \"y\": (\"leaf\", [])}\n        names = [\"x\", \"y\"]\n        for k in range(30):\n            op = rng.choice([\"add\", \"mul\"])\n            G[f\"n{k}\"] = (op, [rng.choice(names), rng.choice(names)])\n            names.append(f\"n{k}\")\n        out = names[-1]\n        f = lambda x, y: ns[\"forward\"](G, {\"x\": x, \"y\": y})[out]\n        v = need(ns, \"forward\")(G, {\"x\": 0.9, \"y\": 1.05})\n        gr = need(ns, \"backward\")(G, v, out)\n        h = 1e-6\n        num = [(f(0.9 + h, 1.05) - f(0.9 - h, 1.05)) / (2 * h), (f(0.9, 1.05 + h) - f(0.9, 1.05 - h)) / (2 * h)]\n        close([gr.get(\"x\", 0.0), gr.get(\"y\", 0.0)], num, rtol=1e-4)\n    R.test(\"30 düğümlü rastgele DAG = sayısal türev\", deep)\n",
 "c": "import numpy as np\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    Var = need(ns, \"Var\", \"sınıf\")\n\n    def grads():\n        x, w = Var([0.5, -1.0, 0.2]), Var([2.0, 3.0, -1.0])\n        need(ns, \"backward\")(need(ns, \"sum_\")(need(ns, \"exp\")(need(ns, \"mul\")(x, w))))\n        e = np.exp(x.data * w.data)\n        close(x.grad, e * w.data); close(w.grad, e * x.data)\n    R.test(\"sum(exp(x·w)) gradyanları\", grads)\n\n    def acc():\n        x, w = Var([1.0]), Var([2.0])\n        for _ in range(2):\n            need(ns, \"backward\")(ns[\"sum_\"](ns[\"mul\"](x, w)))\n        close(x.grad, [4.0], what=\"iki backward → birikim\")\n    R.test(\"yapraklarda gradyan birikimi\", acc)\n\n    def bad_exp():\n        x = Var([0.3, 0.7])\n        y = need(ns, \"exp\")(x)\n        need(ns, \"add_\")(y, 1.0)\n        try:\n            need(ns, \"backward\")(ns[\"sum_\"](y))\n        except RuntimeError as e:\n            ensure(\"inplace\" in str(e), \"hata mesajı 'inplace' içermeli\")\n            return \"exp çıktısını sakladığı için hata: doğru\"\n        raise AssertionError(\"exp çıktısı in-place değişti, backward hata vermeliydi\")\n    R.test(\"exp(x).add_(1) → hata\", bad_exp)\n\n    def ok_mul():\n        x, w = Var([0.3, 0.7]), Var([2.0, -1.0])\n        y = need(ns, \"mul\")(x, w)\n        need(ns, \"add_\")(y, 1.0)\n        need(ns, \"backward\")(ns[\"sum_\"](y))\n        close(x.grad, w.data); close(w.grad, x.data)\n    R.test(\"mul(x, w).add_(1) → sorun yok\", ok_mul)\n\n    def bad_mul():\n        x, w = Var([0.3, 0.7]), Var([2.0, -1.0])\n        h = need(ns, \"exp\")(x)\n        z = need(ns, \"mul\")(h, w)\n        need(ns, \"add_\")(h, 1.0)\n        try:\n            need(ns, \"backward\")(ns[\"sum_\"](z))\n        except RuntimeError:\n            return\n        raise AssertionError(\"mul'un sakladığı girdi değişti, hata bekleniyordu\")\n    R.test(\"saklanan girdi değişirse → hata\", bad_mul)\n",
 "d": "import torch\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    def gn():\n        x = torch.randn(4, requires_grad=True); w = torch.randn(4, requires_grad=True)\n        h = x * w\n        y = (h.exp() + h.sin() * x).sum()\n        nodes = need(ns, \"graph_nodes\")(y)\n        ensure(len(nodes) == len(set(nodes)), \"tekrar eden düğüm var\")\n        ensure(nodes[0] is y.grad_fn, \"ilk düğüm çıktının grad_fn'i\")\n        pos = {n: i for i, n in enumerate(nodes)}\n        for n in nodes:\n            for c, _ in n.next_functions:\n                if c is not None:\n                    ensure(c in pos and pos[n] < pos[c], f\"{type(n).__name__}, {type(c).__name__}'den önce gelmeli\")\n        names = sorted(type(n).__name__ for n in nodes)\n        ensure(names.count(\"AccumulateGrad\") == 2, \"iki yaprak → iki AccumulateGrad\")\n        return \" → \".join(type(n).__name__ for n in nodes)\n    R.test(\"graph_nodes: topolojik ve tam\", gn)\n\n    def ss():\n        a, W = torch.randn(64, 128, requires_grad=True), torch.randn(128, 128, requires_grad=True)\n        ensure(tuple(need(ns, \"saved_stats\")(lambda a, W: torch.tanh(a @ W), a, W)) == (3, 131072), \"tanh(a @ W): 3 tensör, 131072 bayt\")\n        n, b = ns[\"saved_stats\"](lambda a: (a * 2).sum(), a)\n        ensure(n == 0, f\"skaler ile çarpma ve sum hiçbir şey saklamamalı; {n} sayıldı\")\n        n, b = ns[\"saved_stats\"](lambda a: a.exp().exp(), a)\n        ensure(n == 2 and b == 2 * 64 * 128 * 4, \"iki exp → iki çıktı saklanır\")\n    R.test(\"saved_stats\", ss)\n\n    def hk():\n        x = torch.tensor([3.0, 4.0], requires_grad=True)\n        h = x * 1.0\n        need(ns, \"clip_grad_hook\")(h, 1.0)\n        (h * torch.tensor([3.0, 4.0])).sum().backward()\n        close(x.grad, [0.6, 0.8], what=\"norm 5 → 1'e kırpıldı\")\n        x.grad = None\n        h = x * 1.0\n        ns[\"clip_grad_hook\"](h, 10.0)\n        (h * torch.tensor([3.0, 4.0])).sum().backward()\n        close(x.grad, [3.0, 4.0], what=\"eşiğin altında dokunulmaz\")\n    R.test(\"clip_grad_hook\", hk)\n\n    def handle():\n        x = torch.ones(2, requires_grad=True)\n        h = x * 1.0\n        hd = need(ns, \"clip_grad_hook\")(h, 0.1)\n        hd.remove()\n        (h * 10).sum().backward()\n        close(x.grad, [10.0, 10.0], what=\"kaldırılan hook etkisiz\")\n    R.test(\"handle.remove()\", handle)\n",
 "e": "import torch\nfrom labcheck import close, need, ensure\n\n\ndef saved_bytes(fn, *args):\n    tot = [0]\n\n    def outer_pack(t):\n        tot[0] += t.numel() * t.element_size()\n        return t\n    with torch.autograd.graph.saved_tensors_hooks(outer_pack, lambda t: t):\n        out = fn(*args)\n    return out, tot[0]\n\n\ndef check(ns, R):\n    torch.manual_seed(0)\n    mlp = torch.nn.Sequential(torch.nn.Linear(256, 512), torch.nn.GELU(), torch.nn.Linear(512, 512), torch.nn.GELU(), torch.nn.Linear(512, 10))\n    x = torch.randn(256, 256)\n\n    def fwd():\n        y_ref = mlp(x)\n        y = need(ns, \"bf16_saved\")(mlp, x)\n        close(y, y_ref, what=\"ileri geçiş değişmemeli\")\n    R.test(\"ileri geçiş aynı\", fwd)\n\n    def grads():\n        mlp.zero_grad(); mlp(x).pow(2).mean().backward()\n        ref = [p.grad.clone() for p in mlp.parameters()]\n        mlp.zero_grad(); need(ns, \"bf16_saved\")(mlp, x).pow(2).mean().backward()\n        rel = max(((p.grad - r).norm() / r.norm()).item() for p, r in zip(mlp.parameters(), ref))\n        ensure(0 < rel < 2e-2, f\"en büyük göreli gradyan hatası {rel:.2e}\")\n        return f\"en büyük göreli gradyan hatası {rel:.2e}\"\n    R.test(\"gradyanlar yakın (ama birebir değil)\", grads)\n\n    def mem():\n        _, full = saved_bytes(mlp, x)\n        tot = [0]\n\n        def count(fn):\n            orig = torch.autograd.graph.saved_tensors_hooks\n\n            class Wrap(orig):\n                def __init__(self, pack, unpack):\n                    def p(t):\n                        r = pack(t)\n                        flat = r if isinstance(r, (tuple, list)) else (r,)\n                        tot[0] += sum(v.numel() * v.element_size() for v in flat if isinstance(v, torch.Tensor))\n                        return r\n                    super().__init__(p, unpack)\n            torch.autograd.graph.saved_tensors_hooks = Wrap\n            try:\n                fn()\n            finally:\n                torch.autograd.graph.saved_tensors_hooks = orig\n        count(lambda: need(ns, \"bf16_saved\")(mlp, x))\n        ensure(tot[0] > 0, \"saved_tensors_hooks kullanılmalı\")\n        ensure(abs(tot[0] / full - 0.5) < 0.02, f\"saklanan bayt oranı {tot[0] / full:.3f}; ~0.5 bekleniyordu\")\n        R.table(\"Saklanan tensörler (5 katmanlı MLP, B = 256)\", [\"\", \"bayt\"],\n                [[\"fp32\", f\"{full:,}\"], [\"bf16 paket\", f\"{tot[0]:,}\"]],\n                note=\"Not: parametreler de (matmul girdisi olarak) saklananlar arasında sayılır; pratikte yalnız aktivasyonları paketlemek için boyuta göre filtrelenir.\")\n    R.test(\"saklanan bellek yarıya iniyor\", mem)\n"
}

In [ ]:
import importlib, labcheck, checks_m6_1_autograd_internals as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

PyTorch grafı **çalışırken** kurar: `requires_grad` bir tensörü kullanan her işlem, çıktıya bir `grad_fn` düğümü bağlar. Düğümler `next_functions` ile girdilerinin düğümlerine işaret eder; yaprak tensörlerin düğümü `AccumulateGrad`'dir ve gradyanı `.grad`'a **ekler**.

`backward()`:

1. Çıktıdan başlayıp grafı topolojik sırayla gezer (her düğüm, tüm tüketicileri işlendikten sonra).
2. Her düğümde gelen gradyanları toplar, hook varsa uygular, kendi türevini çarpıp girdilere dağıtır.
3. Yapraklarda `.grad += …`.

Türevi hesaplamak için düğümler bazı tensörleri **saklar** (saved tensors): `mul` iki girdisini, `exp` ve `tanh` çıktılarını, `matmul` iki girdisini. Eğitimdeki aktivasyon belleğinin büyük kısmı bunlardır.

Saklanan bir tensör sonradan in-place değiştirilirse geri geçiş yanlış olur. PyTorch bunu version counter ile yakalar: her in-place işlem tensörün sürümünü artırır; düğüm sakladığı sürümü geri geçişte karşılaştırır.

## 1 · Elle

<!-- çözüm -->

Hook bir **düğümün toplam gradyanını** değiştirir, sonra o değer girdilere dağılır. $a$'nın iki yolu var: $d=c+a$ doğrudan (hook'tan etkilenmez) ve $c=ab$ üzerinden (etkilenir).

Saklanan tensör sorusunda `W` bir parametre; yine de matmul onu saklar (referans olarak: kopya değil, ama sayılan bellek o tensörün kendisi).


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**$a=2$, $b=3$: $c=ab$, $d=c+a$, $L=d^2$**

- `da`: $\partial L/\partial a$
- `db`: $\partial L/\partial b$
- `dah`: $c$'ye gradyanı 2 ile çarpan hook varken $\partial L/\partial a$
- `dbh`: Hook varken $\partial L/\partial b$
- `acc`: Hook'suz, `zero_grad` olmadan iki kez `backward`: `a.grad`

**Saklanan tensörler: `torch.tanh(a @ W)`, `a` (64, 128), `W` (128, 128), fp32**

- `nsave`: Kaç tensör saklanır? (matmul iki girdisini, tanh çıktısını saklar)
- `bytes`: Toplam bayt

In [ ]:
cevaplar = dict(
    da=None,
    db=None,
    dah=None,
    dbh=None,
    acc=None,
    nsave=None,
    bytes=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"da":[64,0],"db":[32,0],"dah":[112,0],"dbh":[64,0],"acc":[128,0],"nsave":[3,0],"bytes":[131072,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python. Graf: `{ad: (işlem, [girdi_adları])}`; işlemler `"leaf"`, `"add"`, `"mul"`, `"square"`.

- `forward(graph, inputs)` → her düğümün değeri (sözlük).
- `toposort(graph, out)` → `out` ile biten liste; her düğüm girdilerinden sonra gelir (DFS post-order).
- `backward(graph, vals, out, hooks=None)` → **yalnız yapraklar** için gradyan sözlüğü. Ters topolojik sırada gez; her düğümün toplam gradyanına (varsa) `hooks[ad](g)` uygula, sonra girdilere dağıt.

In [ ]:
%%writefile stage_b.py
def forward(graph, inputs):
    ...


def toposort(graph, out):
    ...


def backward(graph, vals, out, hooks=None):
    ...


g = {"a": ("leaf", []), "b": ("leaf", []), "c": ("mul", ["a", "b"]), "d": ("add", ["c", "a"]), "L": ("square", ["d"])}
vals = forward(g, {"a": 2.0, "b": 3.0})
print(backward(g, vals, "L"))                                # {'a': 64, 'b': 32}
print(backward(g, vals, "L", hooks={"c": lambda x: 2 * x}))  # {'a': 112, 'b': 64}

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy ile minik bir autograd; amaç PyTorch'un in-place güvenliğini taklit etmek.

`Var(data)` hazır: `data` (ndarray), `version` (0), `grad`, `parents`, `_back` (geri fonksiyonu). Sen yaz:

- `mul(a, b)` — **girdileri** saklar: `(a, a.version)`, `(b, b.version)`.
- `exp(a)` — **çıktısını** saklar.
- `sum_(a)` — skaler.
- `add_(a, c)` — `a.data += c` (yerinde), `a.version += 1`. Grafı değiştirmez (sabit eklemek türevi değiştirmez).
- `backward(out)` — topolojik sıra, yapraklarda `grad` biriktir. Geri fonksiyonu sakladığı tensörün sürümü değişmişse `RuntimeError("... modified by an inplace operation")` fırlatsın.

In [ ]:
%%writefile stage_c.py
import numpy as np


class Var:
    def __init__(self, data, parents=(), back=None):
        self.data = np.asarray(data, dtype=float)
        self.version = 0
        self.grad = None
        self.parents = parents
        self._back = back      # g -> girdilerin gradyanları (parents sırasıyla)


# ---- senin kodun ----

def mul(a, b):
    ...


def exp(a):
    ...


def sum_(a):
    ...


def add_(a, c):
    ...


def backward(out):
    ...


x, w = Var([0.5, -1.0]), Var([2.0, 3.0])
s = sum_(exp(mul(x, w)))
backward(s)
print(x.grad, w.grad)

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

- `graph_nodes(t)` → `t.grad_fn`'den erişilebilen tüm düğümler (tekrarsız, `None` hariç); her düğüm `next_functions`'taki düğümlerden **önce** gelir.
- `saved_stats(fn, *args)` → `(sayı, bayt)`: `fn(*args)` çalışırken saklanan tensörler. `torch.autograd.graph.saved_tensors_hooks(pack, unpack)`: `pack` her saklanan tensör için çağrılır.
- `clip_grad_hook(t, max_norm)` — `t.register_hook` ile, `t`'ye gelen gradyanın normu `max_norm`'u aşarsa ölçekleyip küçült; handle'ı döndür.

In [ ]:
%%writefile stage_d.py
import torch


def graph_nodes(t):
    ...


def saved_stats(fn, *args):
    ...


def clip_grad_hook(t, max_norm):
    ...


x = torch.randn(3, requires_grad=True); w = torch.randn(3, requires_grad=True)
y = (x * w).exp().sum()
print([type(n).__name__ for n in graph_nodes(y)])
a, W = torch.randn(64, 128, requires_grad=True), torch.randn(128, 128, requires_grad=True)
print(saved_stats(lambda a, W: torch.tanh(a @ W), a, W))   # (3, 131072)

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

`bf16_saved(fn, *args)` — `fn(*args)`'ı, saklanan tüm kayan noktalı tensörleri bf16'ya **paketleyerek** çalıştır, çıktıyı döndür. `pack`: `(t.dtype, t.to(torch.bfloat16))` (kayan nokta değilse dokunma); `unpack`: orijinal dtype'a geri çevir.

Bağlamı `torch.autograd.graph.saved_tensors_hooks(pack, unpack)` diye tam adıyla çağır (kontrol bu çağrıyı izleyerek paketlenen baytı ölçüyor). Kontrol, saklanan baytın yarıya indiğini, ileri geçişin değişmediğini ve gradyanların yakın kaldığını ölçer.

In [ ]:
%%writefile stage_e.py
import torch


def bf16_saved(fn, *args):
    """fn(*args); saklanan kayan noktalı tensörler bf16'da tutulur."""
    ...


torch.manual_seed(0)
mlp = torch.nn.Sequential(torch.nn.Linear(256, 512), torch.nn.GELU(), torch.nn.Linear(512, 256))
x = torch.randn(128, 256)
bf16_saved(mlp, x).pow(2).mean().backward()
print(mlp[0].weight.grad.norm())

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Maclaurin, Duvenaud, Adams (2015): tersinir öğrenmeyle hiperparametre optimizasyonu

[Maclaurin, Duvenaud, Adams — Gradient-based Hyperparameter Optimization through Reversible Learning (ICML 2015)](https://arxiv.org/abs/1502.03492)

Tüm eğitim sürecinin (T adım SGD + momentum) doğrulama kaybına göre öğrenme oranı ve başlatma ölçeğinin gradyanını hesapla: önce saf autograd ile tüm iteratları saklayarak (bellek O(T)), sonra makalenin tersinir öğrenmesiyle (Algoritma 2: momentumlu SGD'yi geriye çalıştır, kaybolan bitleri sabit noktalı aritmetikle sakla). İki yöntemin hipergradyanlarını karşılaştır ve küçük bir ağda öğrenme oranı takvimini hipergradyanla optimize et (Şekil 2).


**Adımlar**

1. İlk sürümde create_graph=True ile adım adım grafiği koru.

**Kabul ölçütleri**

- İki yöntemin hipergradyanları uyumlu.
- Öğrenilmiş takvim grafiği ve bellek karşılaştırması.

## Kaynaklar

- [PyTorch Docs — Autograd mechanics](https://docs.pytorch.org/docs/stable/notes/autograd.html) — BSD-3-Clause
- [PyTorch Tutorials — Hooks for autograd saved tensors](https://docs.pytorch.org/tutorials/intermediate/autograd_saved_tensors_hooks_tutorial.html) — BSD-3-Clause
- [Edward Z. Yang — PyTorch internals](http://blog.ezyang.com/2019/05/pytorch-internals/) — ©
- [Paszke ve ark. — Automatic differentiation in PyTorch](https://openreview.net/forum?id=BJJsrmfCZ) — OpenReview